# Breast cancer stage classification from gene expression — reproducible analysis

Code accompanying the manuscript:

**An Interpretable Machine Learning Framework for Early- versus Late-Stage Classification of Breast Cancer from Gene Expression, with Co-expression Network Analysis**
Sriporna Biswas and Tanjidul Huda.

This notebook reproduces every analysis reported in the manuscript: cohort definition (Table 1, Figure 2), classifier
performance (Table 2, Figures 3–4), the class-weighting sensitivity analysis (Table S3), permutation importance (Figure 5),
differential expression (Table 3, Table S4, Figure 6), the co-expression network (Figures 7–8) and the workflow figure
(Figure 1). The final cell checks the key numbers against the values reported in the manuscript.

**How to run**
1. Install the pinned dependencies: `pip install -r requirements.txt` (Python 3.13 was used).
2. Put the UCSC Xena download `denseDataOnlyDownload-3.tsv` in `data/` (see `data/README.md`), or set the
   environment variable `BRCA_DATA` to its path. In Google Colab you are asked to upload it if it is missing.
3. Run all cells. Outputs (CSV tables and 300-dpi figures) are written to `results/`.

**Fixed implementation choices.** The manuscript results were produced with a built-in SMOTE implementation, scikit-learn's
`HistGradientBoostingClassifier` and permutation importance. This notebook always uses exactly these components, so the results
do not depend on whether `imbalanced-learn`, `xgboost` or `shap` happen to be installed.

In [ ]:
# ===================== 1. Settings =====================
import os, sys, json, math, hashlib, warnings

DATA_FILE = os.environ.get("BRCA_DATA", os.path.join("data", "denseDataOnlyDownload-3.tsv"))
EXPECTED_SHA256 = "491c44d60cbfa3ae2dd89b5d151679cd6dee425fad7724aee6f6b2ec7a8c7b2e"  # file used for the manuscript
SEED = 42
OUT = "results"

In [ ]:
# ===================== 2. Imports =====================
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.patches import FancyBboxPatch, Patch
from scipy import stats
from scipy.cluster.hierarchy import linkage, leaves_list
import sklearn
from sklearn.base import BaseEstimator
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline as _SkPipeline
from sklearn.utils.metaestimators import available_if
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import (train_test_split, StratifiedKFold, GridSearchCV,
                                     cross_val_score, cross_val_predict)
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, confusion_matrix,
                             balanced_accuracy_score, recall_score, precision_score,
                             f1_score, matthews_corrcoef, accuracy_score)
import networkx as nx

warnings.filterwarnings("ignore")
os.makedirs(OUT, exist_ok=True)
np.random.seed(SEED)


class SMOTE(BaseEstimator):
    """SMOTE (Chawla et al., 2002): interpolate between minority samples and their k nearest minority neighbours."""
    def __init__(self, random_state=None, k_neighbors=5):
        self.random_state = random_state; self.k_neighbors = k_neighbors

    def fit_resample(self, X, y):
        X = np.asarray(X, dtype=float); y = np.asarray(y); rng = np.random.default_rng(self.random_state)
        cls, cnt = np.unique(y, return_counts=True); mino = cls[np.argmin(cnt)]; need = cnt.max() - cnt.min()
        Xm = X[y == mino]
        nn = NearestNeighbors(n_neighbors=min(self.k_neighbors + 1, len(Xm))).fit(Xm)
        idx = nn.kneighbors(Xm, return_distance=False)
        a = rng.integers(0, len(Xm), need); b_ = idx[a, rng.integers(1, idx.shape[1], need)]
        new = Xm[a] + rng.random((need, 1)) * (Xm[b_] - Xm[a])
        return np.vstack([X, new]), np.concatenate([y, np.full(need, mino)])


class Pipeline(_SkPipeline):
    """Pipeline whose resampling steps act only during fit (same semantics as imblearn.pipeline.Pipeline)."""
    def fit(self, X, y=None, **kw):
        Xt, yt = X, y
        for _, st in self.steps[:-1]:
            if hasattr(st, "fit_resample"): Xt, yt = st.fit_resample(Xt, yt)
            else: Xt = st.fit_transform(Xt, yt)
        self.steps[-1][1].fit(Xt, yt); return self

    def _tx(self, X):
        for _, st in self.steps[:-1]:
            if not hasattr(st, "fit_resample"): X = st.transform(X)
        return X

    def predict_proba(self, X, **kw): return self.steps[-1][1].predict_proba(self._tx(X))
    def predict(self, X, **kw): return self.steps[-1][1].predict(self._tx(X))

    @available_if(lambda self: hasattr(self.steps[-1][1], "decision_function"))
    def decision_function(self, X, **kw): return self.steps[-1][1].decision_function(self._tx(X))

    @property
    def classes_(self): return self.steps[-1][1].classes_


BOOST = "Gradient Boosting"
EARLY_C, LATE_C = "#2a78d6", "#eb6834"
MODEL_C = {"Logistic Regression": "#2a78d6", "SVM (RBF)": "#eb6834", "Random Forest": "#1baf7a", BOOST: "#6250d6"}
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9, "axes.titlesize": 10,
    "axes.labelsize": 9, "axes.edgecolor": INK2, "axes.labelcolor": INK,
    "xtick.color": INK2, "ytick.color": INK2, "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "legend.frameon": False, "savefig.dpi": 300,
    "savefig.bbox": "tight", "lines.linewidth": 2,
})
R = {"versions": {"python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
                  "scipy": __import__("scipy").__version__, "scikit-learn": sklearn.__version__,
                  "matplotlib": matplotlib.__version__, "networkx": nx.__version__}}
print(json.dumps(R["versions"]))

In [ ]:
# ===================== 3. Locate and check the input file =====================
if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files   # only available in Google Colab
        print("Please upload denseDataOnlyDownload-3.tsv")
        DATA_FILE = list(files.upload().keys())[0]
    except ImportError:
        raise FileNotFoundError(f"Input file not found: {DATA_FILE}. See data/README.md, or set BRCA_DATA to the file path.")
sha = hashlib.sha256(open(DATA_FILE, "rb").read()).hexdigest()
print("SHA-256:", sha)
if sha != EXPECTED_SHA256:
    print("WARNING: this is not the exact file used for the manuscript; results may differ.")

In [ ]:
# ===================== 4. Load data and define the cohort =====================
EARLY = {"Stage I", "Stage IA", "Stage IB", "Stage II", "Stage IIA", "Stage IIB"}
LATE = {"Stage III", "Stage IIIA", "Stage IIIB", "Stage IIIC", "Stage IV"}

raw = pd.read_csv(DATA_FILE, sep="\t")
id_col = "sample" if "sample" in raw.columns else raw.columns[0]
stage_col = "pathologic_stage"
assert stage_col in raw.columns, "column 'pathologic_stage' not found"
meta_cols = [c for c in [id_col, "samples", stage_col] if c in raw.columns]
GENES_ALL = [c for c in raw.columns if c not in meta_cols]

raw[id_col] = raw[id_col].astype(str)
raw["sample_type"] = raw[id_col].str[13:15]
R["n_raw"] = int(len(raw))
R["sample_types"] = {str(k): int(v) for k, v in raw["sample_type"].value_counts().items()}

d = raw[raw["sample_type"] == "01"].copy()            # primary solid tumours only
R["n_primary"] = int(len(d))
d["patient"] = d[id_col].str[:12]
d = d.drop_duplicates("patient")                      # one sample per patient
R["n_primary_unique"] = int(len(d))

d["stage"] = d[stage_col].fillna("Missing").astype(str).str.strip()
d["y"] = d["stage"].map(lambda s: 0 if s in EARLY else (1 if s in LATE else np.nan))
R["excluded_stage"] = {k: int(v) for k, v in d.loc[d["y"].isna(), "stage"].value_counts().items()}
d = d.dropna(subset=["y"]).reset_index(drop=True)
y = d["y"].astype(int).values

X = d[GENES_ALL].apply(pd.to_numeric, errors="coerce")
miss = X.isna().mean()
GENES = [g for g in GENES_ALL if miss[g] <= 0.20]
R["genes_dropped_missing"] = [g for g in GENES_ALL if g not in GENES]
X = X[GENES]
R["missing_cells_pct"] = round(100 * float(X.isna().values.mean()), 2)

R["n"] = int(len(y)); R["n_early"] = int((y == 0).sum()); R["n_late"] = int((y == 1).sum())
R["pct_late"] = round(100 * y.mean(), 1); R["k"] = len(GENES)
R["stage_detail"] = {k: int(v) for k, v in d["stage"].value_counts().items()}
print(json.dumps({k: R[k] for k in ["n_raw", "sample_types", "n_primary", "n_primary_unique",
                                     "excluded_stage", "n", "n_early", "n_late", "pct_late", "k"]}, indent=1))

# Figure 2: stage distribution
order = ["Stage I", "Stage IA", "Stage IB", "Stage II", "Stage IIA", "Stage IIB",
         "Stage III", "Stage IIIA", "Stage IIIB", "Stage IIIC", "Stage IV"]
cnt = [(s, R["stage_detail"].get(s, 0)) for s in order if R["stage_detail"].get(s, 0) > 0]
fig, ax = plt.subplots(figsize=(6.4, 2.8))
bars = ax.bar([s.replace("Stage ", "") for s, _ in cnt], [c for _, c in cnt], width=0.7,
              color=[EARLY_C if s in EARLY else LATE_C for s, _ in cnt], edgecolor="white", linewidth=1)
for b, (_, c) in zip(bars, cnt):
    ax.text(b.get_x() + b.get_width() / 2, c, str(c), ha="center", va="bottom", fontsize=7.5, color=INK2)
ax.set_xlabel("AJCC pathologic stage"); ax.set_ylabel("Primary tumours")
ax.grid(axis="x", visible=False)
ax.legend(handles=[Patch(color=EARLY_C, label=f"Early, I–II (n = {R['n_early']})"),
                   Patch(color=LATE_C, label=f"Late, III–IV (n = {R['n_late']})")], loc="upper right")
fig.savefig(f"{OUT}/fig2_stage_distribution.png"); plt.close(fig)

In [ ]:
# ===================== 5. Train/test split, tuning, thresholds and evaluation =====================
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
R["n_train"], R["n_test"] = int(len(y_tr)), int(len(y_te))
R["n_train_early"], R["n_train_late"] = int((y_tr == 0).sum()), int((y_tr == 1).sum())
R["n_test_early"], R["n_test_late"] = int((y_te == 0).sum()), int((y_te == 1).sum())
R["acc_all_early_test"] = round(100 * (y_te == 0).mean(), 1)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

MODELS = {
    "Logistic Regression": (LogisticRegression(max_iter=5000, random_state=SEED),
                            {"clf__C": [0.001, 0.01, 0.1, 1, 10]}),
    "SVM (RBF)": (SVC(kernel="rbf", probability=True, random_state=SEED),
                  {"clf__C": [0.1, 1, 10], "clf__gamma": ["scale", 0.01, 0.001]}),
    "Random Forest": (RandomForestClassifier(n_estimators=500, random_state=SEED, n_jobs=-1),
                      {"clf__max_depth": [None, 5, 10], "clf__min_samples_leaf": [1, 5]}),
    BOOST: (HistGradientBoostingClassifier(random_state=SEED, early_stopping=False),
            {"clf__max_iter": [200, 400], "clf__max_depth": [2, 3, 5], "clf__learning_rate": [0.03, 0.1]}),
}

def youden(yt, p):
    fpr, tpr, thr = roc_curve(yt, p)
    return float(thr[np.argmax(tpr - fpr)])

def metrics_at(yt, p, t):
    pr = (p >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(yt, pr, labels=[0, 1]).ravel()
    return {"ROC-AUC": roc_auc_score(yt, p), "PR-AUC": average_precision_score(yt, p),
            "Balanced accuracy": balanced_accuracy_score(yt, pr),
            "Sensitivity": recall_score(yt, pr, zero_division=0),
            "Specificity": tn / (tn + fp) if tn + fp else 0.0,
            "Precision": precision_score(yt, pr, zero_division=0),
            "F1": f1_score(yt, pr, zero_division=0), "MCC": matthews_corrcoef(yt, pr),
            "Accuracy": accuracy_score(yt, pr), "Threshold": t,
            "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)}

def boot_ci(yt, p, t, n=1000):
    rng = np.random.default_rng(SEED); a, b = [], []
    yt, p = np.asarray(yt), np.asarray(p)
    for _ in range(n):
        i = rng.integers(0, len(yt), len(yt))
        if len(np.unique(yt[i])) < 2: continue
        a.append(roc_auc_score(yt[i], p[i])); b.append(balanced_accuracy_score(yt[i], (p[i] >= t).astype(int)))
    return np.percentile(a, [2.5, 97.5]), np.percentile(b, [2.5, 97.5])

# DeLong test for two correlated ROC curves (DeLong et al., 1988; fast algorithm of Sun & Xu, 2014)
def _midrank(x):
    j = np.argsort(x); z = x[j]; n = len(x); t = np.zeros(n); i = 0
    while i < n:
        k = i
        while k < n and z[k] == z[i]: k += 1
        t[i:k] = 0.5 * (i + k - 1) + 1; i = k
    out = np.empty(n); out[j] = t; return out

def delong_test(yt, p1, p2):
    yt = np.asarray(yt); o = np.argsort(-yt, kind="mergesort")
    P = np.vstack([np.asarray(p1)[o], np.asarray(p2)[o]]); m = int(yt.sum()); n = len(yt) - m
    tx = np.array([_midrank(r[:m]) for r in P]); ty = np.array([_midrank(r[m:]) for r in P])
    tz = np.array([_midrank(r) for r in P])
    aucs = tz[:, :m].sum(1) / m / n - (m + 1.0) / (2.0 * n)
    v01 = (tz[:, :m] - tx) / n; v10 = 1.0 - (tz[:, m:] - ty) / m
    S = np.cov(v01) / m + np.cov(v10) / n
    var = S[0, 0] + S[1, 1] - 2 * S[0, 1]
    z = (aucs[0] - aucs[1]) / math.sqrt(var) if var > 0 else 0.0
    return float(aucs[0]), float(aucs[1]), float(2 * stats.norm.sf(abs(z)))

def make_pipe(est, smote=True):
    steps = [("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]
    if smote: steps.append(("smote", SMOTE(random_state=SEED, k_neighbors=5)))
    steps.append(("clf", est))
    return Pipeline(steps)

FIT, PROB, ROWS, PARAMS, ROWS_CW = {}, {}, [], {}, []
for name, (est, grid) in MODELS.items():
    # primary analysis: SMOTE inside the training folds
    gs = GridSearchCV(make_pipe(est), grid, scoring="roc_auc", cv=cv, n_jobs=-1).fit(X_tr, y_tr)
    best = gs.best_estimator_
    cv_auc = cross_val_score(best, X_tr, y_tr, cv=cv, scoring="roc_auc")
    oof = cross_val_predict(best, X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]
    t = youden(y_tr, oof)
    p = best.predict_proba(X_te)[:, 1]
    m = metrics_at(y_te, p, t)
    (alo, ahi), (blo, bhi) = boot_ci(y_te, p, t)
    m.update({"AUC_lo": alo, "AUC_hi": ahi, "BA_lo": blo, "BA_hi": bhi,
              "CV_AUC_mean": cv_auc.mean(), "CV_AUC_sd": cv_auc.std(), "Model": name})
    ROWS.append(m); FIT[name] = best; PROB[name] = p
    PARAMS[name] = {"grid": {k.replace("clf__", ""): v for k, v in grid.items()},
                    "best": {k.replace("clf__", ""): v for k, v in gs.best_params_.items()}}
    # sensitivity analysis: class weighting instead of SMOTE
    est_cw = MODELS[name][0].__class__(**MODELS[name][0].get_params())
    est_cw.set_params(class_weight="balanced")
    gs_cw = GridSearchCV(make_pipe(est_cw, smote=False), grid, scoring="roc_auc", cv=cv, n_jobs=-1).fit(X_tr, y_tr)
    oof_cw = cross_val_predict(gs_cw.best_estimator_, X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]
    p_cw = gs_cw.best_estimator_.predict_proba(X_te)[:, 1]
    mc = metrics_at(y_te, p_cw, youden(y_tr, oof_cw)); mc["Model"] = name; ROWS_CW.append(mc)
    print(f"{name:20s} test AUC {m['ROC-AUC']:.3f} [{alo:.3f}-{ahi:.3f}]  BA {m['Balanced accuracy']:.3f}  "
          f"Sens {m['Sensitivity']:.3f} Spec {m['Specificity']:.3f} | CV AUC {cv_auc.mean():.3f}±{cv_auc.std():.3f}")

RES = pd.DataFrame(ROWS).set_index("Model")
RES_CW = pd.DataFrame(ROWS_CW).set_index("Model")
BEST = RES["ROC-AUC"].idxmax()        # model with the highest test AUC
R["best"] = BEST
R["delong"] = {}
for name in RES.index:
    if name != BEST:
        _, _, pv = delong_test(y_te, PROB[BEST], PROB[name]); R["delong"][name] = pv
R["params"] = PARAMS
RES.round(4).to_csv(f"{OUT}/table2_test_performance.csv")
RES_CW.round(4).to_csv(f"{OUT}/tableS3_class_weight_no_smote.csv")
pd.DataFrame([{"Model": k, "Grid": json.dumps(v["grid"]), "Selected": json.dumps(v["best"])} for k, v in PARAMS.items()]
             ).to_csv(f"{OUT}/tableS2_hyperparameters.csv", index=False)
print("\nHighest test ROC-AUC:", BEST, "| DeLong p vs others:", {k: round(v, 3) for k, v in R["delong"].items()})

In [ ]:
# ===================== 6. ROC / PR curves (Figure 3) and confusion matrix (Figure 4) =====================
fig, axs = plt.subplots(1, 2, figsize=(7.2, 3.4))
for name in RES.sort_values("ROC-AUC", ascending=False).index:
    fpr, tpr, _ = roc_curve(y_te, PROB[name]); pr, rc, _ = precision_recall_curve(y_te, PROB[name])
    axs[0].plot(fpr, tpr, color=MODEL_C[name], label=f"{name} ({RES.loc[name, 'ROC-AUC']:.3f})")
    axs[1].plot(rc, pr, color=MODEL_C[name], label=f"{name} ({RES.loc[name, 'PR-AUC']:.3f})")
axs[0].plot([0, 1], [0, 1], ls="--", lw=1, color=INK2)
axs[1].axhline(y_te.mean(), ls="--", lw=1, color=INK2)
axs[0].set(xlabel="False positive rate (1 − specificity)", ylabel="True positive rate (sensitivity)",
           title="(a) ROC curves, AUC in brackets", xlim=(0, 1), ylim=(0, 1.01))
axs[1].set(xlabel="Recall (sensitivity)", ylabel="Precision", title="(b) Precision–recall, AP in brackets",
           xlim=(0, 1), ylim=(0, 1.01))
for a in axs: a.legend(fontsize=7, loc="lower right" if a is axs[0] else "upper right"); a.set_aspect("equal")
fig.tight_layout(); fig.savefig(f"{OUT}/fig3_roc_pr.png"); plt.close(fig)

mb = RES.loc[BEST]
cm = np.array([[mb["TN"], mb["FP"]], [mb["FN"], mb["TP"]]]).astype(int)
fig, ax = plt.subplots(figsize=(3.2, 3.0))
ax.imshow(cm, cmap="Blues"); ax.grid(False)
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i, j]}\n({100 * cm[i, j] / cm[i].sum():.0f}%)", ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else INK, fontsize=10)
ax.set_xticks([0, 1], ["Early", "Late"]); ax.set_yticks([0, 1], ["Early", "Late"])
ax.set_xlabel("Predicted stage"); ax.set_ylabel("True stage")
ax.set_title(f"{BEST}, threshold {mb['Threshold']:.2f}")
fig.savefig(f"{OUT}/fig4_confusion_matrix.png"); plt.close(fig)

In [ ]:
# ===================== 7. Permutation importance for the Gradient Boosting model (Figure 5) =====================
gb_pipe = FIT[BOOST]
pi = permutation_importance(gb_pipe, X_te, y_te, scoring="roc_auc", n_repeats=30, random_state=SEED)
imp = pd.Series(np.clip(pi.importances_mean, 0, None), index=GENES)
p_hi = gb_pipe.predict_proba(X_te)[:, 1]
direction = pd.Series([stats.spearmanr(X_te[g], p_hi, nan_policy="omit")[0] for g in GENES], index=GENES)
top = imp.sort_values().tail(15)
fig, ax = plt.subplots(figsize=(6.4, 4.6))
ax.barh(top.index, top.values, color=[LATE_C if direction[g] > 0 else EARLY_C for g in top.index], height=0.6)
ax.set_xlabel("Mean decrease in test ROC-AUC when permuted"); ax.grid(axis="y", visible=False)
fig.savefig(f"{OUT}/fig5_permutation_importance.png"); plt.close(fig)
imp = imp.sort_values(ascending=False)
R["top_importance"] = list(imp.index[:10])
R["importance_direction"] = {g: ("late" if direction[g] > 0 else "early") for g in imp.index[:10]}
pd.DataFrame({"importance": imp, "higher_expression_pushes_towards": [("late" if direction[g] > 0 else "early") for g in imp.index]}
             ).to_csv(f"{OUT}/permutation_importance.csv")
print("Top genes:", R["top_importance"])

In [ ]:
# ===================== 8. Differential expression, early vs late (Table 3, Table S4, Figure 6) =====================
de = []
Xe, Xl = X[y == 0], X[y == 1]       # all 1,075 analysed tumours
for g in GENES:
    a, b = Xe[g].dropna(), Xl[g].dropna()
    u, p = stats.mannwhitneyu(b, a, alternative="two-sided")
    de.append({"Gene": g, "Mean early": a.mean(), "Mean late": b.mean(),
               "Difference (late − early)": b.mean() - a.mean(),
               "Effect size r": 2 * u / (len(a) * len(b)) - 1,   # rank-biserial; > 0 means higher in late stage
               "p": p})
DE = pd.DataFrame(de)
pv = DE["p"].values; o = np.argsort(pv); q = np.empty_like(pv)
q[o] = np.minimum.accumulate((pv[o] * len(pv) / np.arange(1, len(pv) + 1))[::-1])[::-1]   # Benjamini–Hochberg
DE["FDR"] = np.minimum(q, 1); DE = DE.sort_values("p").reset_index(drop=True)
DE.to_csv(f"{OUT}/tableS4_differential_expression.csv", index=False)
SIG = DE[DE["FDR"] < 0.05]
R["n_sig"] = int(len(SIG)); R["n_nominal_p05"] = int((DE["p"] < 0.05).sum())
print("Genes with FDR < 0.05:", R["n_sig"], "| nominal p < 0.05:", R["n_nominal_p05"])

hm_genes = list(DE["Gene"].head(20))
Zall = (X[hm_genes] - X[hm_genes].mean()) / X[hm_genes].std()
Zall = Zall.fillna(0).clip(-3, 3)
ordr = np.argsort(y, kind="stable")
fig = plt.figure(figsize=(7.0, 4.4))
ax0 = fig.add_axes([0.14, 0.86, 0.72, 0.035]); ax1 = fig.add_axes([0.14, 0.08, 0.72, 0.77])
cax = fig.add_axes([0.88, 0.08, 0.02, 0.4])
ax0.imshow(y[ordr][None, :], aspect="auto", cmap=matplotlib.colors.ListedColormap([EARLY_C, LATE_C]))
ax0.set_xticks([]); ax0.set_yticks([0], ["Stage"]); ax0.grid(False)
for s in ax0.spines.values(): s.set_visible(False)
ax0.text(R["n_early"] / 2, -0.9, "Early (I–II)", ha="center", fontsize=8, color=INK)
ax0.text(R["n_early"] + R["n_late"] / 2, -0.9, "Late (III–IV)", ha="center", fontsize=8, color=INK)
im = ax1.imshow(Zall.values[ordr].T, aspect="auto", cmap="RdBu_r", vmin=-3, vmax=3, interpolation="nearest")
ax1.set_yticks(range(len(hm_genes)), hm_genes, fontsize=7); ax1.set_xticks([]); ax1.grid(False)
ax1.set_xlabel(f"Primary tumours (n = {R['n']}), sorted by stage group")
ax1.axvline(R["n_early"] - 0.5, color=INK, lw=1)
fig.colorbar(im, cax=cax).set_label("Expression z-score", fontsize=8)
fig.savefig(f"{OUT}/fig6_heatmap.png"); plt.close(fig)

In [ ]:
# ===================== 9. Co-expression network on training tumours (Figures 7–8) =====================
C = X_tr.corr(method="pearson")
Lk = linkage(C.fillna(0).values, "average"); lo = leaves_list(Lk)
Co = C.iloc[lo, lo]
fig, ax = plt.subplots(figsize=(7.0, 6.2))
im = ax.imshow(Co.values, cmap="RdBu_r", vmin=-1, vmax=1); ax.grid(False)
ax.set_xticks(range(len(Co)), Co.columns, rotation=90, fontsize=5.5)
ax.set_yticks(range(len(Co)), Co.index, fontsize=5.5)
fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02).set_label("Pearson r", fontsize=8)
fig.savefig(f"{OUT}/fig7_correlation.png"); plt.close(fig)

NET = {}
for thr in (0.7, 0.5, 0.3):
    G = nx.Graph(); G.add_nodes_from(GENES)
    for i, a in enumerate(GENES):
        for b in GENES[i + 1:]:
            r = C.loc[a, b]
            if pd.notna(r) and abs(r) >= thr: G.add_edge(a, b, r=float(r))
    NET[thr] = G
R["net"] = {str(t): {"edges": NET[t].number_of_edges(),
                     "nodes": sum(1 for _, k in NET[t].degree() if k > 0)} for t in NET}
R["edges07"] = [(a, b, round(dd["r"], 2)) for a, b, dd in NET[0.7].edges(data=True)]
HUB_T = 0.5 if NET[0.5].number_of_edges() >= 10 else 0.3     # prespecified fallback rule
R["hub_thr"] = HUB_T
Gh = NET[HUB_T]
deg = pd.Series(dict(Gh.degree())).sort_values(ascending=False)
deg = deg[deg > 0]
R["hubs"] = [(g, int(k)) for g, k in deg.head(10).items()]
pd.DataFrame({"degree": deg}).to_csv(f"{OUT}/hub_genes.csv")

fig, axs = plt.subplots(1, 2, figsize=(7.6, 4.0), gridspec_kw={"width_ratios": [1.5, 1]})
Gs = Gh.subgraph([g for g in Gh if Gh.degree(g) > 0]).copy()
if Gs.number_of_nodes():
    order_nodes = [g for g in Co.columns if g in Gs]
    ang = {g: 2 * math.pi * i / len(order_nodes) for i, g in enumerate(order_nodes)}
    pos = {g: (math.cos(a), math.sin(a)) for g, a in ang.items()}
    ecol = [LATE_C if dd["r"] > 0 else EARLY_C for _, _, dd in Gs.edges(data=True)]
    nx.draw_networkx_edges(Gs, pos, ax=axs[0], edge_color=ecol, width=[0.6 + 2 * (abs(dd["r"]) - HUB_T) for _, _, dd in Gs.edges(data=True)], alpha=0.7)
    hubset = set(g for g, _ in R["hubs"][:5])
    nx.draw_networkx_nodes(Gs, pos, ax=axs[0], node_size=[20 + 12 * Gs.degree(g) for g in Gs],
                           node_color=["#fcd9c6" if g in hubset else "#dfe9f7" for g in Gs], edgecolors=INK2, linewidths=0.6)
    for g, a in ang.items():
        deg_a = math.degrees(a); flip = 90 < deg_a < 270
        axs[0].text(1.13 * math.cos(a), 1.13 * math.sin(a), g, fontsize=6.3, color=INK,
                    rotation=deg_a + 180 if flip else deg_a, rotation_mode="anchor",
                    ha="right" if flip else "left", va="center",
                    fontweight="bold" if g in hubset else "normal")
    axs[0].set_xlim(-1.55, 1.55); axs[0].set_ylim(-1.55, 1.55); axs[0].set_aspect("equal")
axs[0].set_title(f"(a) Co-expression network, |r| ≥ {HUB_T}"); axs[0].axis("off")
top10 = deg.head(10)[::-1]
axs[1].barh(top10.index, top10.values, color=LATE_C, height=0.6)
for i, v in enumerate(top10.values): axs[1].text(v, i, f" {v}", va="center", fontsize=7.5, color=INK2)
axs[1].set_xlabel("Degree (number of co-expressed partners)"); axs[1].set_title("(b) Top hub genes")
axs[1].grid(axis="y", visible=False)
fig.tight_layout(); fig.savefig(f"{OUT}/fig8_network_hubs.png"); plt.close(fig)
print("Network:", R["net"], "| hub threshold:", HUB_T, "| hubs:", R["hubs"][:5])

In [ ]:
# ===================== 10. Workflow figure (Figure 1) =====================
def workflow_fig(path):
    fig, ax = plt.subplots(figsize=(7.2, 6.0)); ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.axis("off")
    def box(x, y0, w, h, txt, hi=False):
        ax.add_patch(FancyBboxPatch((x, y0), w, h, boxstyle="round,pad=0.3,rounding_size=1.5",
                     fc="#dfe9f7" if hi else "white", ec=EARLY_C if hi else INK2, lw=1.4 if hi else 0.9))
        ax.text(x + w / 2, y0 + h / 2, txt, ha="center", va="center", fontsize=7, color=INK, linespacing=1.35)
    def arr(x1, y1, x2, y2):
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="-|>", color=INK2, lw=0.9))
    box(12, 91, 76, 7, f"TCGA-BRCA expression + clinical data (UCSC Xena)\n{R['k']} cancer-related genes")
    box(12, 79, 76, 7, f"Primary tumours, one per patient\nearly I–II (n = {R['n_early']})  vs  late III–IV (n = {R['n_late']})")
    box(12, 67, 76, 7, f"Stratified 80/20 split\ntraining n = {R['n_train']}  |  held-out test n = {R['n_test']}", hi=True)
    box(1, 34, 47, 26, f"Training set, stratified 5-fold CV\nmedian imputation → z-scaling\n→ SMOTE (training folds only)\nLR | SVM | RF | {BOOST}\ngrid search on ROC-AUC\nYouden threshold from\nout-of-fold predictions")
    box(52, 34, 47, 26, "Biological analyses\nearly vs late differential expression\n(Mann–Whitney, BH-FDR)\nco-expression network\n(training samples) → hub genes")
    box(1, 15, 47, 12, "Single evaluation on the test set\nROC-AUC (95% CI), PR-AUC, balanced\naccuracy, sensitivity, specificity, MCC;\nDeLong test")
    box(1, 1, 47, 8, f"Permutation-importance explanation of the\n{BOOST} model on the test set")
    box(52, 12, 47, 15, "Integration\ngenes ranked by importance, differential\nexpression and network degree")
    arr(50, 91, 50, 86.4); arr(50, 79, 50, 74.4); arr(40, 67, 24.5, 60.4); arr(60, 67, 75.5, 60.4)
    arr(24.5, 34, 24.5, 27.4); arr(24.5, 15, 24.5, 9.4); arr(75.5, 34, 75.5, 27.4); arr(48.3, 5, 55, 11.6)
    fig.savefig(path); plt.close(fig)
workflow_fig(f"{OUT}/fig1_workflow.png")

with open(f"{OUT}/results.json", "w") as fh:
    json.dump(R, fh, indent=1, default=str)
print("All analyses done. Outputs are in", OUT)

In [ ]:
# ===================== 11. Check against the values reported in the manuscript =====================
expected = {
    "analysed tumours": (R["n"], 1075), "early": (R["n_early"], 805), "late": (R["n_late"], 270),
    "training": (R["n_train"], 860), "test": (R["n_test"], 215),
    "LR ROC-AUC": (round(RES.loc["Logistic Regression", "ROC-AUC"], 3), 0.618),
    "LR AUC CI low": (round(RES.loc["Logistic Regression", "AUC_lo"], 3), 0.530),
    "LR AUC CI high": (round(RES.loc["Logistic Regression", "AUC_hi"], 3), 0.704),
    "LR balanced accuracy": (round(RES.loc["Logistic Regression", "Balanced accuracy"], 3), 0.578),
    "LR sensitivity": (round(RES.loc["Logistic Regression", "Sensitivity"], 3), 0.741),
    "LR specificity": (round(RES.loc["Logistic Regression", "Specificity"], 3), 0.416),
    "SVM ROC-AUC": (round(RES.loc["SVM (RBF)", "ROC-AUC"], 3), 0.595),
    "RF ROC-AUC": (round(RES.loc["Random Forest", "ROC-AUC"], 3), 0.589),
    "GB ROC-AUC": (round(RES.loc[BOOST, "ROC-AUC"], 3), 0.594),
    "DeLong p vs SVM": (round(R["delong"]["SVM (RBF)"], 3), 0.199),
    "class-weighted LR ROC-AUC (Table S3)": (round(RES_CW.loc["Logistic Regression", "ROC-AUC"], 3), 0.651),
    "genes with FDR < 0.05": (R["n_sig"], 0), "CD44 FDR": (round(float(DE.loc[DE.Gene == "CD44", "FDR"].iloc[0]), 3), 0.057),
    "top importance gene": (R["top_importance"][0], "POU5F1"), "top hub": (tuple(R["hubs"][0]), ("REL", 11)),
    "edges |r|>=0.5": (R["net"]["0.5"]["edges"], 70),
}
ok = True
for k, (got, exp) in expected.items():
    flag = "OK " if got == exp else "DIFF"; ok &= got == exp
    print(f"{flag}  {k}: {got} (manuscript {exp})")
print("\nALL CHECKS PASSED" if ok else "\nSOME VALUES DIFFER FROM THE MANUSCRIPT")